# Install Depedensi

In [ ]:
!pip install Sastrawi
!pip install tqdm
! pip install joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.2 MB/s eta 0:00:00


# Import Library

In [ ]:
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import re
from tqdm.auto import tqdm
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, make_scorer, f1_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')


# Read Datasets

In [ ]:
data_intensi = pd.read_csv('dataset_balanced_intensi_final.csv')
data_intensi.head(3)

,full_text,topic,type
0,ya kalo nyuruh pake gojek dari awal juga udah ...,gojek,komplain
1,mini gratitude minggu akhir juni ini; foto mir...,gojek,pujian
2,Lu kira gojek kaahh pake dikasih bintang rat,gojek,pernyataan


In [ ]:
data_intensi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   full_text        6000 non-null   object
 1   topic            6000 non-null   object
 2   intention        6000 non-null   object
 3   clean_text       6000 non-null   object
 4   normalized_text  6000 non-null   object
 5   no_stopword      6000 non-null   object
 6   stemmed_text     6000 non-null   object
dtypes: object(7)
memory usage: 328.3+ KB


In [ ]:
print(f"jumlah baris dan kolom : {data_intensi.shape}")

jumlah baris dan kolom : (6000, 3)


In [ ]:
data_intensi.columns = ["full_text" , "topic" , "intention"]

# Exploratory Data Analysis

### missing value

In [ ]:
data_intensi.isna().sum()

,0
full_text,0
topic,0
intention,0


In [ ]:
baris_missing = data_intensi[data_intensi['intention'].isnull()]

print(baris_missing)

Empty DataFrame
Columns: [full_text, topic, intention]
Index: []


### distribution class

In [ ]:
count_df = data_intensi.value_counts("intention")
count_df_frame = count_df.reset_index()
df = count_df_frame.sort_values('count', ascending=False)
styled_df = df.style.background_gradient(
    subset=['count'],
    cmap='Blues',
    low=0.3,
    high=0.8
).set_properties(
    subset=['intention', 'count'], **{'text-align': 'center', 'font-weight': 'bold'}
)
styled_df

,intention,count
0,komplain,1500
1,pernyataan,1500
2,pertanyaan,1000
3,pujian,1000
4,saran-kritik,1000


In [ ]:
import plotly.express as px

intention_counts = data_intensi['intention'].value_counts().reset_index()
intention_counts.columns = ['intention', 'count']

fig = px.bar(
    intention_counts,
    x='intention',
    y='count',
    color='count',
    title='Distribution of Intention'
)

fig.update_layout(
    xaxis_title='Intention',
    yaxis_title='Count',
    title_x=0.5,
    font=dict(size=14),
    plot_bgcolor='white',
    bargap=0.2
)
fig.update_xaxes(tickangle=45)
fig.show()


### Wordcloud

In [ ]:
from wordcloud import WordCloud
import plotly.express as px
import matplotlib.pyplot as plt

text = " ".join(data_intensi['full_text'])
wc = WordCloud(
    width=1600,
    height=800,
    background_color='white',
    colormap='viridis',
    max_words=300,
    min_font_size=10,
    max_font_size=200
).generate(text)

wc.to_file("wordcloud.png")

fig = px.imshow(plt.imread("wordcloud.png"))
fig.update_layout(
    title="WordCloud of Text Data",
    title_x=0.5,
    xaxis_visible=False,
    yaxis_visible=False,
    dragmode=False,
    plot_bgcolor='white'
)
fig.show()


# Pre - Processing

### normalized

In [ ]:
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        emoji_pattern = re.compile(
            "["
            u"\U0001F600-\U0001F64F"
            u"\U0001F300-\U0001F5FF"
            u"\U0001F680-\U0001F6FF"
            u"\U0001F1E0-\U0001F1FF"
            u"\U00002700-\U000027BF"
            u"\U000024C2-\U0001F251"
            "]+", flags=re.UNICODE)
        text = emoji_pattern.sub(r'', text)
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(r'@\w+|#\w+', '', text)
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
    else:
        text = ''
    return text
data_intensi['clean_text'] = data_intensi['full_text'].apply(clean_text)
data_intensi[['full_text', 'clean_text']].head()


,full_text,clean_text
0,ya kalo nyuruh pake gojek dari awal juga udah ...,ya kalo nyuruh pake gojek dari awal juga udah ...
1,mini gratitude minggu akhir juni ini; foto mir...,mini gratitude minggu akhir juni ini foto mirr...
2,Lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake dikasih bintang rat
3,naik gojek yang ngebut ngebut dan songgong ban...,naik gojek yang ngebut ngebut dan songgong ban...
4,Tolong dong gosend perbaiki sistem petanya ser...,tolong dong gosend perbaiki sistem petanya ser...


In [ ]:
import re

ABBREV_MAP = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "ngga": "tidak", "tdk": "tidak",
    "t": "tidak", "tak": "tidak", "enggak": "tidak", "ndak": "tidak", "nda": "tidak",

    "yg": "yang", "yng": "yang", "dgn": "dengan", "dg": "dengan", "dng": "dengan",
    "utk": "untuk", "buat": "untuk", "bwt": "untuk", "buad": "untuk",
    "sm": "sama", "sma": "sama", "ama": "sama", "ma": "sama",
    "aja": "saja", "aj": "saja",
    "jd": "jadi", "jdi": "jadi", "jdih": "jadi",
    "pd": "pada", "pda": "pada",
    "krn": "karena", "karn": "karena", "karna": "karena", "krena": "karena",
    "dr": "dari", "drpd": "daripada", "drpdnya": "daripadanya",
    "trus": "terus", "trs": "terus", "teruss": "terus",
    "tp": "tapi", "tpi": "tapi",
    "bkn": "bukan", "bkan": "bukan",
    "blm": "belum", "blom": "belum",
    "sdh": "sudah", "udh": "sudah", "udah": "sudah", "ud": "sudah", "suda": "sudah", "dh":'sudah',
    "dlm": "dalam", "dalem":"dalam", "dlam":"dalam", "skr": "sekarang", "skg": "sekarang", "skrg":"sekarang",
    "kt": "kita",
    "km": "kamu", "kmu": "kamu", "kam": "kamu", "kamuuh": "kamu",
    "sy": "saya", "aq": "saya", "q": "saya", "gue": "saya", "gw": "saya", "gua": "saya", "gwe": "saya",
    "lg": "lagi", "lgi": "lagi",
    "bgt": "banget", "bgtt": "banget", "bngt": "banget", "bangt": "banget",
    "brp": "berapa", "brapa": "berapa",
    "hrs": "harus", "harusny": "harusnya",
    "kl": "kalau", "klo": "kalau", "klu": "kalau", "kalo": "kalau",
    "kmrn": "kemarin", "kmren": "kemarin",
    "bs": "bisa", "bsa": "bisa", "bsq": "bisa",
    "mo": "mau", "mw": "mau", "mow": "mau", "mwu": "mau",
    "plis": "tolong", "pls": "tolong", "tolonglah": "tolong",
    "trmksih": "terima kasih", "makasih": "terima kasih", "makasii": "terima kasih",
    "makasihh": "terima kasih", "thx": "terima kasih", "thanks": "terima kasih",
    "ok": "oke", "okay": "oke", "okey": "oke", "okayy": "oke", "okee": "oke",
    "bbrp": "beberapa", "bbrapa": "beberapa",
    "org": "orang", "orng": "orang", "org2": "orang-orang", "orng2": "orang-orang",
    "mls": "malas", "males": "malas",
    "ksl": "kesal", "kesel": "kesal",
    "tmn": "teman", "temen": "teman", "tmn2": "teman-teman",
    "smg": "semoga",
    "insyaallah": "insya allah", "inshaallah": "insya allah",
    "astgfirullah": "astaghfirullah", "astaghfirulah": "astaghfirullah",
    "alhamdulilah": "alhamdulillah", "alhamdulila": "alhamdulillah",
    "bismilah": "bismillah", "bismila": "bismillah",
    "amiin": "amin", "aminn": "amin", "aminnn": "amin",
    "gituu": "begitu", "gituuu": "begitu", "gituan": "begituan",
    "nih": "ini", "nie": "ini", "ni": "ini",
    "tu": "itu", "tuh": "itu", "ituu": "itu",
    "ajaib": "ajaib",
    "bnr": "benar", "bener": "benar",
    "bgtu": "begitu",
    "bnyk": "banyak", "byk": "banyak",
    "btw": "ngomong-ngomong",
    "jg": "juga", "jga": "juga",
    "skrng": "sekarang",
    "cm": "cuma", "cma": "cuma", "cuman": "cuma",
    "bsk": "besok", "bsok": "besok",
    "dpt": "dapat", "dapet": "dapat",
    "tmpt": "tempat",
    "trs": "terus",
    "td": "tadi",
    "tpn": "tapi",
    "lbh": "lebih",
    "krg": "kurang",
    "cr":"cari", "cri":"cari",
    "dptin": "dapatkan",
    "bisaaa": "bisa",
    "bnget": "banget",

    "ppk": "pepek", "ngentod": "ngentot", "anj": "anjir",
    "anjir": "anjir", "anjay": "anjir", "bjir": "anjir",
    "jir": "anjir", "bejir": "anjir", "anjg": "anjing",
    "njir": "anjir", "njay": "anjir",
    "asu": "anjing", "asw": "anjing", "anjeng":"anjing",
    "bbi":"babi", "mnyt":"monyet","kntl":"kontol",

    "woww": "wow", "wkwk": "haha", "wkwkwk": "haha", "wkww": "haha",
    "hehe": "haha", "hihi": "haha", "hehehe": "haha", "ahah": "haha",
    "yaa": "ya", "yah": "ya", "yaaa": "ya",
    "lho": "loh", "loh": "loh", "kok": "kenapa",
    "dongg": "dong", "donk": "dong",
    "deh": "deh", "lahh": "lah",
    "siih": "sih", "sihh": "sih",
    "bangettt": "banget", "parahh": "parah",
    "mantul": "mantap betul", "mantapp": "mantap", "mantappu": "mantap",
    "kece": "keren", "ciamik": "bagus",
    "bt": "bad mood", "bete": "bad mood",
    "gabisa": "tidak bisa", "gapapa": "tidak apa-apa", "gpp": "tidak apa-apa",
    "gajadi": "tidak jadi", "gaboleh": "tidak boleh",
    "nggaada": "tidak ada", "gaada": "tidak ada", "gakada": "tidak ada",
    "jln":"jalan", "plsss":"please", "grab express" : "grabexpress", "grab car": "grabcar", "ajg" : "anjing",
    "emg":"memang","emng":"memang",
    "loch":"loh", "suru" : "suruh", "aje": "aja", "banggg":"bang",
    "takutttttt":"takut", "blg":"bilang", "blng":"bilang", "knp":"kenapa"
}

def normalize_abbrev(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    normalized_words = [ABBREV_MAP.get(w.lower(), w) for w in words]
    return " ".join(normalized_words)

data_intensi['normalized_text'] = data_intensi['clean_text'].apply(normalize_abbrev)
data_intensi[['full_text', 'normalized_text']].head(5)


,full_text,normalized_text
0,ya kalo nyuruh pake gojek dari awal juga udah ...,ya kalau nyuruh pake gojek dari awal juga suda...
1,mini gratitude minggu akhir juni ini; foto mir...,mini gratitude minggu akhir juni ini foto mirr...
2,Lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake dikasih bintang rat
3,naik gojek yang ngebut ngebut dan songgong ban...,naik gojek yang ngebut ngebut dan songgong ban...
4,Tolong dong gosend perbaiki sistem petanya ser...,tolong dong gosend perbaiki sistem petanya ser...


### Stopword

In [ ]:
data_intensi.head(5)

,full_text,topic,intention,clean_text,normalized_text
0,ya kalo nyuruh pake gojek dari awal juga udah ...,gojek,komplain,ya kalo nyuruh pake gojek dari awal juga udah ...,ya kalau nyuruh pake gojek dari awal juga suda...
1,mini gratitude minggu akhir juni ini; foto mir...,gojek,pujian,mini gratitude minggu akhir juni ini foto mirr...,mini gratitude minggu akhir juni ini foto mirr...
2,Lu kira gojek kaahh pake dikasih bintang rat,gojek,pernyataan,lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake dikasih bintang rat
3,naik gojek yang ngebut ngebut dan songgong ban...,gojek,komplain,naik gojek yang ngebut ngebut dan songgong ban...,naik gojek yang ngebut ngebut dan songgong ban...
4,Tolong dong gosend perbaiki sistem petanya ser...,gosend,saran-kritik,tolong dong gosend perbaiki sistem petanya ser...,tolong dong gosend perbaiki sistem petanya ser...


In [ ]:
factory = StopWordRemoverFactory()
stopword_list = factory.get_stop_words()
stopword_set = set(stopword_list)

def remove_stopwords(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    filtered = [w for w in words if w.lower() not in stopword_set]
    return " ".join(filtered)

data_intensi['no_stopword'] = data_intensi['normalized_text'].apply(remove_stopwords)
data_intensi[['normalized_text', 'no_stopword']].head()

,normalized_text,no_stopword
0,ya kalau nyuruh pake gojek dari awal juga suda...,kalau nyuruh pake gojek awal pesen anjir
1,mini gratitude minggu akhir juni ini foto mirr...,mini gratitude minggu akhir juni foto mirror m...
2,lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake dikasih bintang rat
3,naik gojek yang ngebut ngebut dan songgong ban...,naik gojek ngebut ngebut songgong banget jadi ...
4,tolong dong gosend perbaiki sistem petanya ser...,dong gosend perbaiki sistem petanya sering ban...


### stemming

In [ ]:
data_intensi.head(5)

,full_text,topic,intention,clean_text,normalized_text,no_stopword
0,ya kalo nyuruh pake gojek dari awal juga udah ...,gojek,komplain,ya kalo nyuruh pake gojek dari awal juga udah ...,ya kalau nyuruh pake gojek dari awal juga suda...,kalau nyuruh pake gojek awal pesen anjir
1,mini gratitude minggu akhir juni ini; foto mir...,gojek,pujian,mini gratitude minggu akhir juni ini foto mirr...,mini gratitude minggu akhir juni ini foto mirr...,mini gratitude minggu akhir juni foto mirror m...
2,Lu kira gojek kaahh pake dikasih bintang rat,gojek,pernyataan,lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake dikasih bintang rat
3,naik gojek yang ngebut ngebut dan songgong ban...,gojek,komplain,naik gojek yang ngebut ngebut dan songgong ban...,naik gojek yang ngebut ngebut dan songgong ban...,naik gojek ngebut ngebut songgong banget jadi ...
4,Tolong dong gosend perbaiki sistem petanya ser...,gosend,saran-kritik,tolong dong gosend perbaiki sistem petanya ser...,tolong dong gosend perbaiki sistem petanya ser...,dong gosend perbaiki sistem petanya sering ban...


In [ ]:
tqdm.pandas(desc="Stemming (Sastrawi)")
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def apply_stemming(text):
    if not isinstance(text, str):
        return ""
    return stemmer.stem(text)

data_intensi['stemmed_text'] = data_intensi['no_stopword'].progress_apply(apply_stemming)


Stemming (Sastrawi):   0%|          | 0/6000 [00:00<?, ?it/s]

In [ ]:
data_intensi.to_csv("modelling_data_intention.csv" , index=False)

# Modelling

In [ ]:
data_intensi = pd.read_csv("modelling_data_intention.csv")
data_intensi.head(3)

,full_text,topic,intention,clean_text,normalized_text,no_stopword,stemmed_text
0,ya kalo nyuruh pake gojek dari awal juga udah ...,gojek,komplain,ya kalo nyuruh pake gojek dari awal juga udah ...,ya kalau nyuruh pake gojek dari awal juga suda...,kalau nyuruh pake gojek awal pesen anjir,kalau nyuruh pake gojek awal sen anjir
1,mini gratitude minggu akhir juni ini; foto mir...,gojek,pujian,mini gratitude minggu akhir juni ini foto mirr...,mini gratitude minggu akhir juni ini foto mirr...,mini gratitude minggu akhir juni foto mirror m...,mini gratitude minggu akhir juni foto mirror m...
2,Lu kira gojek kaahh pake dikasih bintang rat,gojek,pernyataan,lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake kasih bintang rat


### Tf-IDF

In [ ]:
tfidf = TfidfVectorizer(
    max_features = 7000,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2)
)
X_tfidf = tfidf.fit_transform(data_intensi['stemmed_text'])
y = data_intensi['intention']

In [ ]:
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 77292 stored elements and shape (6000, 7000)>

In [ ]:
print(f"Shape TF-IDF: {X_tfidf.shape}")
print(f"Jumlah sampel: {X_tfidf.shape[0]}")
print(f"Jumlah fitur TF-IDF: {X_tfidf.shape[1]}")

Shape TF-IDF: (6000, 7000)
Jumlah sampel: 6000
Jumlah fitur TF-IDF: 7000


## 80 / 20

### Split data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(f"Train set: {X_train.shape[0]} sampel")
print(f"Test set: {X_test.shape[0]} sampel")

Train set: 4800 sampel
Test set: 1200 sampel


In [ ]:
df_X_test = pd.DataFrame.sparse.from_spmatrix(X_test)
df_X_test.head(3)

,0,1,2,3,4,5,6,7,8,9,...,6990,6991,6992,6993,6994,6995,6996,6997,6998,6999
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### modelling

In [ ]:
lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

base_models = [
    ('lgbm', lgbm),
    ('xgb', xgb),
]

meta_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    multi_class='multinomial',
    solver='lbfgs'
)

stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1,
)

In [ ]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('lgbm',
                                LGBMClassifier(learning_rate=0.05, max_depth=7,
                                               n_estimators=500,
                                               random_state=42, verbose=-1)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogl...
                                              learning_rate=0.05, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=7,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=500, n_jobs=None,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression(max_iter=1000,
                                                      multi_class='multinomial',
                                                      random_state=42),
                   n_jobs=-1)

### Evaluation

In [ ]:
y_pred_train = stacking_clf.predict(X_train)
y_pred_test = stacking_clf.predict(X_test)

print("\n--- Train Set Performance ---")
train_accuracy = accuracy_score(y_train, y_pred_train)
print(f"Accuracy: {train_accuracy:.4f}")
print("\n--- Test Set Performance ---")
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Accuracy: {test_accuracy:.4f}")
print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_test))
print("\n--- Confusion Matrix (Test Set) ---")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)


--- Train Set Performance ---
Accuracy: 0.9185

--- Test Set Performance ---
Accuracy: 0.7108

--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

    komplain       0.73      0.77      0.75       300
  pernyataan       0.61      0.74      0.67       300
  pertanyaan       0.76      0.70      0.73       200
      pujian       0.78      0.68      0.73       200
saran-kritik       0.77      0.62      0.69       200

    accuracy                           0.71      1200
   macro avg       0.73      0.70      0.71      1200
weighted avg       0.72      0.71      0.71      1200


--- Confusion Matrix (Test Set) ---
[[230  40   8   6  16]
 [ 31 222  17  21   9]
 [ 11  41 141   2   5]
 [ 18  32   7 135   8]
 [ 24  30  13   8 125]]


In [ ]:
import pickle
with open('stacking_model_8020_intention.pkl', 'wb') as f:
    pickle.dump(stacking_clf, f)
print("Model disimpan: stacking_model_8020_intention.pkl")

with open('tfidf_vectorizer8020_intention.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("TF-IDF vectorizer disimpan: tfidf_vectorize8020_intention.pkl")



Model disimpan: stacking_model_8020_intention.pkl
TF-IDF vectorizer disimpan: tfidf_vectorize8020_intention.pkl


## 70 / 30

### split data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
print(f"Train set: {X_train.shape[0]} sampel")
print(f"Test set: {X_test.shape[0]} sampel")

Train set: 4200 sampel
Test set: 1800 sampel


In [ ]:
df_X_test = pd.DataFrame.sparse.from_spmatrix(X_test)
df_X_test.head(3)

,0,1,2,3,4,5,6,7,8,9,...,6990,6991,6992,6993,6994,6995,6996,6997,6998,6999
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### modelling

In [ ]:
lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

base_models = [
    ('lgbm', lgbm),
    ('xgb', xgb),
]

meta_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    multi_class='multinomial',
    solver='lbfgs'
)

stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1,
)

In [ ]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('lgbm',
                                LGBMClassifier(learning_rate=0.05, max_depth=7,
                                               n_estimators=500,
                                               random_state=42, verbose=-1)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogl...
                                              learning_rate=0.05, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=7,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=500, n_jobs=None,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression(max_iter=1000,
                                                      multi_class='multinomial',
                                                      random_state=42),
                   n_jobs=-1)

### Evaluation

In [ ]:
y_pred_train = stacking_clf.predict(X_train)
y_pred_test = stacking_clf.predict(X_test)

print("\n--- Train Set Performance ---")
train_accuracy = accuracy_score(y_train, y_pred_train)
print(f"Accuracy: {train_accuracy:.4f}")
print("\n--- Test Set Performance ---")
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Accuracy: {test_accuracy:.4f}")
print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_test))
print("\n--- Confusion Matrix (Test Set) ---")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)


--- Train Set Performance ---
Accuracy: 0.9250

--- Test Set Performance ---
Accuracy: 0.7028

--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

    komplain       0.73      0.76      0.74       450
  pernyataan       0.58      0.71      0.64       450
  pertanyaan       0.76      0.68      0.71       300
      pujian       0.76      0.69      0.72       300
saran-kritik       0.80      0.64      0.71       300

    accuracy                           0.70      1800
   macro avg       0.73      0.70      0.71      1800
weighted avg       0.71      0.70      0.70      1800


--- Confusion Matrix (Test Set) ---
[[342  62  10  16  20]
 [ 46 321  32  39  12]
 [ 24  65 203   2   6]
 [ 23  53   8 207   9]
 [ 35  49  15   9 192]]


In [ ]:
import pickle
with open('stacking_model_7030_intention.pkl', 'wb') as f:
    pickle.dump(stacking_clf, f)
print("Model disimpan: stacking_model.pkl")

with open('tfidf_vectorizer7030_intention.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("TF-IDF vectorizer disimpan: tfidf_vectorizer7030_intention.pkl")



Model disimpan: stacking_model.pkl
TF-IDF vectorizer disimpan: tfidf_vectorizer7030_intention.pkl


# Hyper Tuning

In [ ]:
import pandas as pd

data_intensi = pd.read_csv("modelling_data_intention.csv")
data_intensi.head(3)

,full_text,topic,intention,clean_text,normalized_text,no_stopword,stemmed_text
0,ya kalo nyuruh pake gojek dari awal juga udah ...,gojek,komplain,ya kalo nyuruh pake gojek dari awal juga udah ...,ya kalau nyuruh pake gojek dari awal juga suda...,kalau nyuruh pake gojek awal pesen anjir,kalau nyuruh pake gojek awal sen anjir
1,mini gratitude minggu akhir juni ini; foto mir...,gojek,pujian,mini gratitude minggu akhir juni ini foto mirr...,mini gratitude minggu akhir juni ini foto mirr...,mini gratitude minggu akhir juni foto mirror m...,mini gratitude minggu akhir juni foto mirror m...
2,Lu kira gojek kaahh pake dikasih bintang rat,gojek,pernyataan,lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake dikasih bintang rat,lu kira gojek kaahh pake kasih bintang rat


### TF - IDF

In [ ]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_tfidf = tfidf.fit_transform(data_intensi['stemmed_text'])
y = data_intensi['intention']


In [ ]:
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 83882 stored elements and shape (6000, 10295)>

### split data 80/20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train set: {X_train.shape[0]} sampel")
print(f"Test set: {X_test.shape[0]} sampel")


Train set: 4800 sampel
Test set: 1200 sampel


In [ ]:
scorer = make_scorer(f1_score, average='weighted')


#### lightgbm

In [ ]:
lgbm_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05],
    'max_depth': [5, -1],
    'num_leaves': [31, 50, 70],
}

lgbm_base = LGBMClassifier(random_state=42, verbose=-1)

lgbm_search = GridSearchCV(
    lgbm_base,
    lgbm_params,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

lgbm_search.fit(X_train, y_train)
best_lgbm = lgbm_search.best_estimator_

# LGBMClassifier(learning_rate=0.05, random_state=42, verbose=-1)


Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [ ]:
print(best_lgbm)

LGBMClassifier(learning_rate=0.05, random_state=42, verbose=-1)


#### xgboost

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

xgb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01,0.05],
    'max_depth': [3, 7, 10],
}

xgb_base = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

xgb_search = GridSearchCV(
    xgb_base,
    xgb_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train_enc)

best_xgb = xgb_search.best_estimator_

# XGBClassifier(base_score=None, booster=None, callbacks=None,
#               colsample_bylevel=None, colsample_bynode=None,
#               colsample_bytree=None, device=None, early_stopping_rounds=None,
#               enable_categorical=False, eval_metric='mlogloss',
#               feature_types=None, feature_weights=None, gamma=None,
#               grow_policy=None, importance_type=None,
#               interaction_constraints=None, learning_rate=0.05, max_bin=None,
#               max_cat_threshold=None, max_cat_to_onehot=None,
#               max_delta_step=None, max_depth=7, max_leaves=None,
#               min_child_weight=None, missing=nan, monotone_constraints=None,
#               multi_strategy=None, n_estimators=300, n_jobs=None,
#               num_parallel_tree=None, ...)

Fitting 3 folds for each of 18 candidates, totalling 54 fits


In [ ]:
print(best_xgb)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)


#### logistic regresion

In [ ]:
lr_params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [500, 1000, 2000]
}

lr_base = LogisticRegression(
    random_state=42,
    multi_class='multinomial'
)

lr_search = GridSearchCV(
    lr_base,
    lr_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

lr_search.fit(X_train, y_train)
best_lr = lr_search.best_estimator_

# LogisticRegression(C=10, max_iter=500, multi_class='multinomial',
#                    random_state=42, solver='saga')

Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [ ]:
print(best_lr)

LogisticRegression(C=10, max_iter=500, multi_class='multinomial',
                   random_state=42, solver='saga')


In [ ]:
# --- Base Models ---
lgbm = LGBMClassifier(
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)

xgb = XGBClassifier(
    learning_rate=0.05,
    max_depth=7,
    n_estimators=300,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)

base_models = [
    ('lgbm', lgbm),
    ('xgb', xgb),
]

# --- Meta Model ---
meta_model = LogisticRegression(
    C=10,
    max_iter=500,
    multi_class='multinomial',
    solver='saga',
    random_state=42
)

# --- Stacking Classifier ---
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

In [ ]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('lgbm',
                                LGBMClassifier(learning_rate=0.05,
                                               random_state=42, verbose=-1)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogloss',
                                              feature_types=None,
                                              featu...
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=7,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=300, n_jobs=None,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression(C=10, max_iter=500,
                                                      multi_class='multinomial',
                                                      random_state=42,
                                                      solver='saga'),
                   n_jobs=-1)

#### evaluation hyper tuning 80 20


In [ ]:
y_pred_train = stacking_clf.predict(X_train)
y_pred_test = stacking_clf.predict(X_test)

print("\n--- Train Set Performance ---")
train_accuracy = accuracy_score(y_train, y_pred_train)
print(f"Accuracy: {train_accuracy:.4f}")
print("\n--- Test Set Performance ---")
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Accuracy: {test_accuracy:.4f}")
print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_test))
print("\n--- Confusion Matrix (Test Set) ---")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)


--- Train Set Performance ---
Accuracy: 0.8829

--- Test Set Performance ---
Accuracy: 0.7075

--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

    komplain       0.74      0.76      0.75       300
  pernyataan       0.59      0.75      0.66       300
  pertanyaan       0.76      0.69      0.72       200
      pujian       0.80      0.70      0.74       200
saran-kritik       0.78      0.60      0.68       200

    accuracy                           0.71      1200
   macro avg       0.73      0.70      0.71      1200
weighted avg       0.72      0.71      0.71      1200


--- Confusion Matrix (Test Set) ---
[[228  43   8   8  13]
 [ 30 224  19  19   8]
 [ 12  46 137   1   4]
 [ 15  32   4 140   9]
 [ 25  35  12   8 120]]


In [ ]:
import pickle
with open('stacking_model_8020_hyper_intention.pkl', 'wb') as f:
    pickle.dump(stacking_clf, f)
print("Model disimpan: stacking_model_8020_hyper_intention.pkl")

with open('tfidf_vectorizer8020_hyper_intention.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("TF-IDF vectorizer disimpan: tfidf_vectorizer8020_hyper_intention.pkl")

Model disimpan: stacking_model_8020_hyper_intention.pkl
TF-IDF vectorizer disimpan: tfidf_vectorizer8020_hyper_intention.pkl


## split data 7030

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(f"Train set: {X_train.shape[0]} sampel")
print(f"Test set: {X_test.shape[0]} sampel")


Train set: 4200 sampel
Test set: 1800 sampel


In [ ]:
scorer = make_scorer(f1_score, average='weighted')


### lightgbm

In [ ]:
lgbm_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05],
    'max_depth': [5, -1],
    'num_leaves': [31, 50, 70],
}

lgbm_base = LGBMClassifier(random_state=42, verbose=-1)

lgbm_search = GridSearchCV(
    lgbm_base,
    lgbm_params,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

lgbm_search.fit(X_train, y_train)
best_lgbm = lgbm_search.best_estimator_


# LGBMClassifier(learning_rate=0.05, random_state=42, verbose=-1)


Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [ ]:
print(best_lgbm)

LGBMClassifier(learning_rate=0.05, random_state=42, verbose=-1)


### xgboost

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

xgb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01,0.05],
    'max_depth': [3, 7, 10],
}

xgb_base = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

xgb_search = GridSearchCV(
    xgb_base,
    xgb_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train_enc)

best_xgb = xgb_search.best_estimator_

# XGBClassifier(base_score=None, booster=None, callbacks=None,
#               colsample_bylevel=None, colsample_bynode=None,
#               colsample_bytree=None, device=None, early_stopping_rounds=None,
#               enable_categorical=False, eval_metric='mlogloss',
#               feature_types=None, feature_weights=None, gamma=None,
#               grow_policy=None, importance_type=None,
#               interaction_constraints=None, learning_rate=0.05, max_bin=None,
#               max_cat_threshold=None, max_cat_to_onehot=None,
#               max_delta_step=None, max_depth=7, max_leaves=None,
#               min_child_weight=None, missing=nan, monotone_constraints=None,
#               multi_strategy=None, n_estimators=300, n_jobs=None,
#               num_parallel_tree=None, ...)

Fitting 3 folds for each of 18 candidates, totalling 54 fits


In [ ]:
print(best_xgb)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)


### logistic regresion

In [ ]:
lr_params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [500, 1000, 2000]
}

lr_base = LogisticRegression(
    random_state=42,
    multi_class='multinomial'
)

lr_search = GridSearchCV(
    lr_base,
    lr_params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

lr_search.fit(X_train, y_train)
best_lr = lr_search.best_estimator_

# LogisticRegression(C=10, max_iter=500, multi_class='multinomial',
#                    random_state=42)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [ ]:
print(best_lr)

LogisticRegression(C=10, max_iter=500, multi_class='multinomial',
                   random_state=42)


In [ ]:
# --- Base Models ---
lgbm = LGBMClassifier(
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)

xgb = XGBClassifier(
    learning_rate=0.05,
    max_depth=7,
    n_estimators=300,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)

base_models = [
    ('lgbm', lgbm),
    ('xgb', xgb),
]

# --- Meta Model ---
meta_model = LogisticRegression(
    C=10,
    max_iter=500,
    multi_class='multinomial',
    random_state=42
)

# --- Stacking Classifier ---
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)


In [ ]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('lgbm',
                                LGBMClassifier(learning_rate=0.05,
                                               random_state=42, verbose=-1)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogloss',
                                              feature_types=None,
                                              featu...
                                              learning_rate=0.05, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=7,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=300, n_jobs=None,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression(C=10, max_iter=500,
                                                      multi_class='multinomial',
                                                      random_state=42),
                   n_jobs=-1)

### evaluation hyper tuning 70 30

In [ ]:
y_pred_train = stacking_clf.predict(X_train)
y_pred_test = stacking_clf.predict(X_test)

print("\n--- Train Set Performance ---")
train_accuracy = accuracy_score(y_train, y_pred_train)
print(f"Accuracy: {train_accuracy:.4f}")
print("\n--- Test Set Performance ---")
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Accuracy: {test_accuracy:.4f}")
print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_test))
print("\n--- Confusion Matrix (Test Set) ---")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)


--- Train Set Performance ---
Accuracy: 0.8886

--- Test Set Performance ---
Accuracy: 0.6972

--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

    komplain       0.73      0.75      0.74       450
  pernyataan       0.58      0.72      0.64       450
  pertanyaan       0.75      0.64      0.69       300
      pujian       0.75      0.71      0.73       300
saran-kritik       0.77      0.63      0.69       300

    accuracy                           0.70      1800
   macro avg       0.72      0.69      0.70      1800
weighted avg       0.71      0.70      0.70      1800


--- Confusion Matrix (Test Set) ---
[[339  59  12  20  20]
 [ 44 322  34  36  14]
 [ 25  68 193   4  10]
 [ 21  48   8 212  11]
 [ 37  54  10  10 189]]


In [ ]:
import pickle
with open('stacking_model_7030_hyper_intention.pkl', 'wb') as f:
    pickle.dump(stacking_clf, f)
print("Model disimpan: stacking_model_7030_hyper_intention.pkl")

with open('tfidf_vectorizer7030_hyper_intention.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("TF-IDF vectorizer disimpan: tfidf_vectorizer7030_hyper_intention.pkl")

Model disimpan: stacking_model_7030_hyper_intention.pkl
TF-IDF vectorizer disimpan: tfidf_vectorizer7030_hyper_intention.pkl
